In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("students_data.team1_taxi.bronze_taxi")
spark.table("students_data.team1_taxi.bronze_taxi").limit(5).display()

In [0]:
silver_clean = (
    bronze_df.select(
        F.col("`Booking ID`").cast("string").alias("booking_id"),
        F.trim(F.col("`Source`")).alias("trip_status"),

        # Timestamps
        F.to_timestamp(F.col("`Pickup Due`"), "dd/MM/yyyy HH:mm").alias("pickup_due_ts"),
        F.to_timestamp(F.col("`Time Dispatched`"), "dd/MM/yyyy HH:mm").alias("time_dispatched_ts"),
        F.to_timestamp(F.col("`Time Vehicle Arrived`"), "dd/MM/yyyy HH:mm").alias("time_vehicle_arrived_ts"),
        F.to_timestamp(F.col("`Time Picked Up`"), "dd/MM/yyyy HH:mm").alias("time_picked_up_ts"),
        F.to_timestamp(F.col("`Completed`"), "dd/MM/yyyy HH:mm").alias("completed_ts"),

        # Just Renaming
        F.trim(F.col("`Driver`")).alias("driver"),
        F.trim(F.col("`Vehicle`")).alias("vehicle"),
        F.trim(F.col("`Payment Type`")).alias("payment_type"),
        F.col("`Priority`").cast("int").alias("priority"),
        F.trim(F.col("`Capabilities`")).alias("capabilities"),
        F.trim(F.col("`Booking source`")).alias("booking_source"),
        # Removing , from price to get rid of casting error
        F.regexp_replace(F.col("`Price`"), ",", "").cast("double").alias("price"),
        F.regexp_replace(F.col("`Distance`"), ",", "").cast("double").alias("distance"),
        F.trim(F.col("`Pickup Zone`")).alias("pickup_zone"),
        F.trim(F.col("`Destination Zone`")).alias("destination_zone"),
        F.col("`Pickup Latitude`").cast("double").alias("pickup_latitude"),
        F.col("`Pickup Longitude`").cast("double").alias("pickup_longitude"),
        F.col("`Destination Latitude`").cast("double").alias("destination_latitude"),
        F.col("`Destination Longitude`").cast("double").alias("destination_longitude"),
        F.trim(F.col("`Booked by`")).alias("booked_by")
    )
)

In [0]:
string_cols = [
    "booking_id", "trip_status", "payment_type",
    "booking_source", "pickup_zone", "destination_zone"
]

for c in string_cols:
    silver_clean = silver_clean.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.col(c))
        # When F.trim(column) is empty set it to None otherwise leave it as it is
    )

In [0]:
# Trips with capability 'Z' should have payment_type set to 'card'
silver_clean = silver_clean.withColumn(
    "payment_type",
    F.when(F.col("capabilities") == "Z", F.lit("card")).otherwise(F.col("payment_type"))
)

In [0]:
# Remove rows where pickup date is in the future (data quality issue)
silver_clean = silver_clean.filter(
    F.col("pickup_due_ts").isNull() | (F.col("pickup_due_ts") <= F.current_timestamp())
)

future_removed = bronze_df.count() - silver_clean.count()
print(f"Rows removed with future pickup dates: {future_removed}")

In [0]:
silver_clean.display()

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("booking_id").orderBy(
    F.col("completed_ts").desc_nulls_last(),
    F.col("time_picked_up_ts").desc_nulls_last(),
    F.col("time_dispatched_ts").desc_nulls_last()
)

silver_dedup = (
    # Rank each row by its booking_id group keep the latest and discard the ranking section.
    silver_clean
    .withColumn("rn", F.row_number().over(w)) 
    .filter(F.col("rn") == 1)
    .drop("rn")
)

silver_dedup.display()

In [0]:
silver_conformed = (
    # Transforming each into a single format with capitals in set points.
    silver_dedup
    .withColumn("payment_type", F.initcap(F.trim(F.col("payment_type"))))
    .withColumn("booking_source", F.initcap(F.trim(F.col("booking_source"))))
    .withColumn("pickup_zone", F.initcap(F.trim(F.col("pickup_zone"))))
    .withColumn("destination_zone", F.initcap(F.trim(F.col("destination_zone"))))
    .withColumn("trip_status", F.initcap(F.trim(F.col("trip_status"))))
)

silver_conformed.display()

In [0]:
(
    # Writing table
    silver_conformed
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("students_data.team1_taxi.silver_taxi")
)